<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Full_LSTM_Project_using_Module_11_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import re, copy
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize
from textblob import TextBlob
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [2]:
nltk.download('punkt')      # tokenize
nltk.download('punkt_tab')

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Modify the dataset by adding your own custom-generated story, paragraph, or personal content to the document.

# You can use your dataset

In [3]:
df = pd.read_csv('/content/IMDB Dataset.csv', encoding = 'latin1', engine = 'python', on_bad_lines='skip')
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


# Preprocess the text data by removing punctuation marks and unnecessary symbols from the documents before tokenization and training.

# Use Regex, regex character set, escape characters, digit character class, wildcard characters asterics, plus quantifiers, question marks, capture and group, findall, search, split, sub, compile, lowercasing, remove html tags, remove url, remove punctuations, handle special texts, spelling mistake correction, stopword removals, remove emojis, spacy and nltk, stemming, lemmitization for preprocess before tokenization.

In [4]:
# Write patterns
html_pattern = re.compile(r"<.*?>")
url_pattern = re.compile(r"https?://\S+|www\.\S+")
emoji_pattern = re.compile(r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF\U00002700-\U000027BF\U000024C2-\U0001F251]+", flags = re.UNICODE)
quote_pattern = re.compile(r"[\"']")
punc_pattern = re.compile(r"[^a-z0-9\s]")
space_pattern = re.compile(r"\s+")

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
Lemmatizer = WordNetLemmatizer()

# create functions
def spell_correct(text):
  return str(TextBlob(text).correct())

def remove_stopwords(text):
  return " ".join(w for w in text.split() if w not in stop_words)

def stem_words(text):
  return " ".join(stemmer.stem(w) for w in text.split())

def lemmatize_words(text):
  return " ".join(Lemmatizer.lemmatize(w, pos = 'v') for w in text.split())

# apply functions
USE_SPELL, USE_STOPWORDS, USE_STEM, USE_LEMMATIZE = False, False, False, True

def tokenize(text):
  text = html_pattern.sub("", text)
  text = url_pattern.sub("", text)
  text = emoji_pattern.sub("", text)
  text = text.lower()

  if USE_SPELL:
    text = spell_correct(text)

  text = quote_pattern.sub('', text)
  text = punc_pattern.sub(' ', text)
  text = space_pattern.sub(' ', text).strip()

  if USE_STOPWORDS:
    text = remove_stopwords(text)

  if USE_STEM:
    text = stem_words(text)

  if USE_LEMMATIZE:
    text = lemmatize_words(text)

  return word_tokenize(text)

df['tokens'] = df['review'].apply(tokenize)
df['tokens']

,tokens
0,"[one, of, the, other, reviewers, have, mention..."
1,"[a, wonderful, little, production, the, film, ..."
2,"[i, think, this, be, a, wonderful, way, to, sp..."
3,"[basically, theres, a, family, where, a, littl..."
4,"[petter, matteis, love, in, the, time, of, mon..."
...,...
49995,"[i, think, this, movie, do, a, down, right, go..."
49996,"[bad, plot, bad, dialogue, bad, act, idiotic, ..."
49997,"[i, be, a, catholic, teach, in, parochial, ele..."
49998,"[im, go, to, have, to, disagree, with, the, pr..."


# Add separate training and testing data loaders. Train the model using the training dataset and evaluate it on the testing dataset.

In [5]:
# Build vocab
vocab = {'<UNK>': 0}

for token, _ in Counter(t for tokens in df['tokens'] for t in tokens).most_common(40000):
  vocab[token] = len(vocab)

# words to numbers
def text_to_indices(tokens, vocab):
  nums = []
  for t in tokens:                                  # loop over the argument
    nums.append(vocab.get(t, 0))
  return nums

MAX_TOKENS = 30                                     # first 30 words of each review
N_REVIEWS = 2000                                    # first 2000 reviews to save memory

reviews = []
for tokens in df['tokens'][:N_REVIEWS]:
  reviews.append(text_to_indices(tokens[:MAX_TOKENS], vocab))

# split by review first, so test sequences come from reviews the model never saw
train_reviews, test_reviews = train_test_split(reviews, test_size=0.2, random_state=42)
train_reviews, val_reviews = train_test_split(train_reviews, test_size=0.1, random_state=42)

# sequences, padding and tensors
max_len = MAX_TOKENS

def make_xy(reviews):
  padded = []
  for nums in reviews:
    for i in range(1, len(nums)):
      s = nums[:i+1]
      padded.append([0] * (max_len - len(s)) + s)
  padded = torch.tensor(padded)
  return padded[:, :-1], padded[:, -1]

X_train, y_train = make_xy(train_reviews)
X_val, y_val = make_xy(val_reviews)
X_test, y_test = make_xy(test_reviews)

print(X_train.shape, X_val.shape, X_test.shape)


# Custom dataset and dataloader
class CustomDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

train_dataset = CustomDataset(X_train, y_train)
val_dataset = CustomDataset(X_val, y_val)
test_dataset = CustomDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size = 128, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 128, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = 128, shuffle = False)

print(f"Training samples: {len(train_dataset)} | Validation samples: {len(val_dataset)} | Testing samples: {len(test_dataset)}")

torch.Size([41742, 29]) torch.Size([4629, 29]) torch.Size([11595, 29])
Training samples: 41742 | Validation samples: 4629 | Testing samples: 11595


In [6]:
# LSTM Model
#  To see overfitting without dropout, change Dropout(0.3) to Dropout(0) and weight_decay to 0, then compare the two accuracies.

class LSTMModel(nn.Module):
  def __init__(self, vocab_size, embed_dim = 100, hidden_dim = 128, dropout = 0.0):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, embed_dim)
    self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first = True)
    self.dropout = nn.Dropout(dropout)
    self.fc = nn.Linear(hidden_dim, vocab_size)

  def forward(self, X):
    embedded = self.embedding(X)
    out, (hidden, cell) = self.lstm(embedded)
    output = self.fc(self.dropout(hidden.squeeze(0)))
    return output

criterion = nn.CrossEntropyLoss()

def get_loss_acc(model, loader):
  model.eval()

  total_loss, correct, total = 0, 0, 0

  with torch.no_grad():
    for batch_x, batch_y in loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)
      output = model(batch_x)
      loss = criterion(output, batch_y)

      total_loss += loss.item() * len(batch_y)
      correct += (output.argmax(dim = 1) == batch_y).sum().item()
      total += len(batch_y)

  return total_loss / total, correct / total * 100


def train(model, optimizer, epochs, patience = None):
  best_loss, wait, best_state = 999, 0, None

  for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)

      optimizer.zero_grad()
      output = model(batch_x)
      loss = criterion(output, batch_y)
      loss.backward()
      optimizer.step()

      total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

    # early stopping: stop when validation loss stops improving
    if patience:
      val_loss, _ = get_loss_acc(model, val_loader)

      if val_loss < best_loss:
        best_loss, wait = val_loss, 0
        best_state = copy.deepcopy(model.state_dict())

      else:
        wait += 1
        if wait >= patience:
          print('Early Stopping')
          break

  if best_state:
    model.load_state_dict(best_state)

# Calculate and compare the accuracy for both the training and testing datasets. Analyze whether the model is suffering from overfitting or not.

In [7]:
# train the baseline model
model = LSTMModel(len(vocab)).to(device)
optimizer = optim.Adam(model.parameters(), lr = 0.001)

train(model, optimizer, epochs = 80)

Epoch 1/80, Loss: 2316.5756
Epoch 2/80, Loss: 2024.8130
Epoch 3/80, Loss: 1936.5533
Epoch 4/80, Loss: 1860.2750
Epoch 5/80, Loss: 1786.2682
Epoch 6/80, Loss: 1712.4839
Epoch 7/80, Loss: 1637.8611
Epoch 8/80, Loss: 1563.9707
Epoch 9/80, Loss: 1489.9077
Epoch 10/80, Loss: 1418.4706
Epoch 11/80, Loss: 1346.9973
Epoch 12/80, Loss: 1279.2647
Epoch 13/80, Loss: 1212.7368
Epoch 14/80, Loss: 1148.8625
Epoch 15/80, Loss: 1090.0510
Epoch 16/80, Loss: 1035.1998
Epoch 17/80, Loss: 985.6080
Epoch 18/80, Loss: 938.5602
Epoch 19/80, Loss: 893.8270
Epoch 20/80, Loss: 851.8391
Epoch 21/80, Loss: 813.1288
Epoch 22/80, Loss: 775.7100
Epoch 23/80, Loss: 740.9981
Epoch 24/80, Loss: 707.4997
Epoch 25/80, Loss: 676.6276
Epoch 26/80, Loss: 646.7971
Epoch 27/80, Loss: 619.0162
Epoch 28/80, Loss: 590.4687
Epoch 29/80, Loss: 565.8712
Epoch 30/80, Loss: 541.7102
Epoch 31/80, Loss: 518.0069
Epoch 32/80, Loss: 496.2729
Epoch 33/80, Loss: 476.0986
Epoch 34/80, Loss: 454.6951
Epoch 35/80, Loss: 436.0168
Epoch 36/80, 

In [8]:
# Evaluate the baseline model
_, train_acc = get_loss_acc(model, train_loader)
_, test_acc = get_loss_acc(model, test_loader)
print(f"Train accuracy: {train_acc:.2f}%")
print(f"Test accuracy:  {test_acc:.2f}%")

if train_acc - test_acc > 10:
    print("The model is OVERFITTING: train accuracy is much higher than test accuracy.")
else:
    print("The model is not showing strong overfitting.")

Train accuracy: 95.42%
Test accuracy:  8.75%
The model is OVERFITTING: train accuracy is much higher than test accuracy.


# If overfitting is observed, apply suitable deep learning techniques discussed in the course, such as dropout, regularization, or early stopping, to reduce it.

# Improve the overall model performance by reducing the training loss and increasing the prediction accuracy through better preprocessing, architecture tuning, hyperparameter adjustment, or optimization techniques.

In [9]:
#Train the improved model
# Dropout, L2 regularization (weight_decay), early stopping, a bigger model and a tuned learning rate.

model2 = LSTMModel(len(vocab), embed_dim = 128, hidden_dim = 256, dropout = 0.3).to(device)
optimizer2 = optim.Adam(model2.parameters(), lr = 0.002, weight_decay=1e-4)
train(model2, optimizer2, epochs=40, patience=3)

Epoch 1/40, Loss: 2279.0116
Epoch 2/40, Loss: 2084.4461
Epoch 3/40, Loss: 1999.5917
Epoch 4/40, Loss: 1937.2325
Epoch 5/40, Loss: 1889.0709
Epoch 6/40, Loss: 1847.0791
Epoch 7/40, Loss: 1808.5623
Epoch 8/40, Loss: 1777.4217
Epoch 9/40, Loss: 1747.9929
Epoch 10/40, Loss: 1722.6007
Epoch 11/40, Loss: 1696.1363
Epoch 12/40, Loss: 1670.6013
Epoch 13/40, Loss: 1649.4304
Epoch 14/40, Loss: 1630.4275
Early Stopping


In [10]:
# evaluate
_, train_acc2 = get_loss_acc(model2, train_loader)
_, test_acc2 = get_loss_acc(model2, test_loader)

print("             Train     Test     Gap")
print(f"Baseline    {train_acc:6.2f}%  {test_acc:6.2f}%  {train_acc - test_acc:6.2f}")
print(f"Improved    {train_acc2:6.2f}%  {test_acc2:6.2f}%  {train_acc2 - test_acc2:6.2f}")

             Train     Test     Gap
Baseline     95.42%    8.75%   86.67
Improved     18.47%   13.99%    4.48


# Extra: Apply sequence padding in the question answering system so that all input sequences have equal length, then train the complete model using the padded dataset.

In [11]:
qa_data = [
    ("where did maya walk", "library"),
    ("what did she find", "lantern"),
    ("what showed the path", "map"),
    ("who smiled at maya", "librarian")
]

# vocab for questions (0 is padding) and for answers
qa_vocab = {'<PAD>': 0}
ans_vocab = {}
for q, a in qa_data:
    for word in tokenize(q):
        if word not in qa_vocab:
            qa_vocab[word] = len(qa_vocab)
    if a not in ans_vocab:
        ans_vocab[a] = len(ans_vocab)

# words to numbers
questions = []
answers = []
for q, a in qa_data:
    questions.append(text_to_indices(tokenize(q), qa_vocab))
    answers.append(ans_vocab[a])

# sequence padding: every question gets the same length
max_q_len = max(len(q) for q in questions)
padded_questions = []
for q in questions:
    padded_questions.append([0] * (max_q_len - len(q)) + q)

X_qa = torch.tensor(padded_questions)
y_qa = torch.tensor(answers)
print(X_qa)

# train on the padded data
qa_dataset = CustomDataset(X_qa, y_qa)
qa_loader = DataLoader(qa_dataset, batch_size=2, shuffle=True)
qa_model = LSTMModel(len(qa_vocab)).to(device)
qa_optimizer = optim.Adam(qa_model.parameters(), lr=0.01)

for epoch in range(50):
    qa_model.train()
    for bx, by in qa_loader:
        bx, by = bx.to(device), by.to(device)
        qa_optimizer.zero_grad()
        loss = criterion(qa_model(bx), by)
        loss.backward()
        qa_optimizer.step()

_, qa_acc = get_loss_acc(qa_model, qa_loader)
print(f"QA accuracy: {qa_acc:.2f}%")

tensor([[ 1,  2,  3,  4],
        [ 5,  2,  6,  7],
        [ 5,  8,  9, 10],
        [11, 12, 13,  3]])
QA accuracy: 100.00%
